# 01b. Precomputed extracellular potentials

In the previous tutorial, we introduced extracellular stimulation and recording in AxonML and the core theoretical concepts. In that tutorial, we computed the field using an analytic model (a point source in an infinite homogenous medium). However, in many cases, the field may be more complex and not amenable to an analytic solution. For example, the field may be generated by a complex electrode geometry, or the medium may be inhomogeneous and anisotropic. In these cases, we can use precomputed fields that are generated from numerical simulations (e.g., finite element method) or from experimental measurements. All the same principles of quasistaticity, supoerposition, and reciprocity still apply, but we need to introduce functionality to consume precomputed field data. In this tutorial, we will show how to use precomputed fields in AxonML.


## Core classes

The core classes for working with precomputed fields are:
- {py:class}`~axonml.models.fields.precomputed_interpolate_1d`: This class implements 1D interpolation of precomputed fields. This is useful for fields that are sampled along a single spatial dimension (e.g., along the length of an axon).
- {py:class}`~axonml.models.fields.precomputed_interpolate_3d_rect`: This class implements 3D interpolation of precomputed fields on a rectilinear grid. This is useful for fields for which you have dense sampling in three dimensions on a consistent grid.
- {py:class}`~axonml.models.fields.precomputed_interpolate_3d_scattered`: This class implements interpolation of precomputed fields at scattered points in 3D space. This is useful for fields for which you have sampling at arbitrary locations in 3D space (e.g., from experimental measurements, or from simulations with irregular / non-rectilinear meshes).
- {py:class}`~axonml.models.fields.efield_interpolate_3d_rect`: This class implements 3D interpolation of precomputed electric fields on a rectilinear grid, and computes the quasipotentials at the model's coordinates. This is useful for computing extracellular stimulation using precomputed E-field data (e.g., in TMS where you solve for the E-field, not potentials).
- {py:class}`~axonml.models.fields.efield_interpolate_3d_scattered`: This class implements interpolation of precomputed electric fields at scattered points in 3D space, and computes the quasipotentials at the model's coordinates. This is useful for computing extracellular stimulation using precomputed E-field data that is sampled at arbitrary locations in 3D space (e.g., from simulations with irregular / non-rectilinear meshes).

In all cases, their use follows the same pattern (as with the analytic field classes):
1. Prepare the field model by providing the precomputed data and sample points.
2. Call the field model on the neuron model to compute the field values at the model's coordinates.
3. Use the computed field values for extracellular stimulation or recording as needed.


## Example: Extracellular stimulation of axon with precomputed fields

We will first simulate a single myelinated axon model receiving extracellular kHz stimulation. For this example, we will assume that we have the field stored as a function of arc length along the axon (e.g., from a 1D cable model of the field along the axon). We will use the {py:class}`~axonml.models.fields.precomputed_interpolate_1d` class to consume this data and compute the potentials at the model's coordinates.

In [ ]:
import numpy as np
import axonml as ax
from axonml.models.mod import hh
from axonml.units import mm, um

model = ax.Myelinated(
    diameters=[6.0], n_node=201, node_length=1.0, celsius=6.3, v_init=-65.0
)
model.insert(hh)

In [ ]:
field_model = ax.precomputed_interpolate_1d(
    data = np.load('0.npy') * 1000, # 2D array of shape (n_fiber_locations, n_points_along_axon), in mV
    x = np.load('fiber_xs.npy'), # 1D array of shape (n_points_along_axon,)
)